<a href="https://colab.research.google.com/github/Ammarah171/NLP/blob/main/Lab4_RNN_Sum'25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module Import

In [ ]:
# !pip install tensorflow -q

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1&2 Load Dataset and Pre-processing

In [ ]:
from datasets import load_dataset

# Load the TweetEval sentiment dataset

dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [ ]:
dataset["train"][0] # 0 = negative, 1 = neutral, 2 = positive

{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"',
 'label': 2}

In [ ]:
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

val_texts = dataset["validation"]["text"]
val_labels = dataset["validation"]["label"]

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Tokenizer

num_words = 10000

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>") # Only keep top 10,000 words
tokenizer.fit_on_texts(train_texts)

In [ ]:
# Convert text to sequences -> 0 = PAD, 1 = OOV, 2 to 9999 = words

X_train = tokenizer.texts_to_sequences(train_texts)
X_val   = tokenizer.texts_to_sequences(val_texts)
X_test  = tokenizer.texts_to_sequences(test_texts)

In [ ]:
X_train[0]

[9020,
 4,
 5,
 2,
 1314,
 1826,
 10,
 2,
 292,
 407,
 7552,
 7553,
 5185,
 2,
 1715,
 10,
 1,
 1]

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad sequences to fixed length as neural networks need inputs to be the same shape (like a matrix).

max_len = 50

X_train = pad_sequences(X_train, maxlen=max_len, padding="post")
X_val   = pad_sequences(X_val, maxlen=max_len, padding="post")
X_test  = pad_sequences(X_test, maxlen=max_len, padding="post")

In [ ]:
X_train[0]

array([9020,    4,    5,    2, 1314, 1826,   10,    2,  292,  407, 7552,
       7553, 5185,    2, 1715,   10,    1,    1,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0], dtype=int32)

In [ ]:
# Labels

y_train = np.array(train_labels)
y_val = np.array(val_labels)
y_test = np.array(test_labels)

In [ ]:
y_train # 0 = negative, 1 = neutral, 2 = positive

array([2, 1, 1, ..., 2, 1, 1])

# 3 Training and Evaluation

## 3.1 Simple RNN & Bidirectional RNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

# Simple RNN Classifier

with tf.device('/GPU:0'):
    model = Sequential([
        Embedding(input_dim=num_words, output_dim=32, input_length=max_len),
        SimpleRNN(32),
        Dropout(0.5), # Dropout for regularization
        Dense(3, activation='softmax') # 3 classes, softmax output
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=64,
        validation_data=(X_val, y_val)
    )

    test_loss, test_acc = model.evaluate(X_test, y_test)
    print("Test Accuracy:", test_acc)

Epoch 1/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.4317 - loss: 1.0332 - val_accuracy: 0.4345 - val_loss: 1.0148
Epoch 2/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.4852 - loss: 0.9867 - val_accuracy: 0.5130 - val_loss: 0.9817
Epoch 3/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.6212 - loss: 0.8341 - val_accuracy: 0.5440 - val_loss: 0.9396
Epoch 4/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7326 - loss: 0.6576 - val_accuracy: 0.5850 - val_loss: 0.9611
Epoch 5/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8048 - loss: 0.5087 - val_accuracy: 0.5600 - val_loss: 1.0588
384/384 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5165 - loss: 1.1847
Test Accuracy: 0.512943685054779


In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 50, 32)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 966,539 (3.69 MB)

 Trainable params: 322,179 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 644,360 (2.46 MB)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Bidirectional

# Simple Bidirectional-RNN Classifier

with tf.device('/GPU:0'):
    model = Sequential([
        Embedding(input_dim=num_words, output_dim=32, input_length=max_len),
        Bidirectional(SimpleRNN(32)),
        Dropout(0.5), # Dropout for regularization
        Dense(3, activation='softmax') # 3 classes, softmax output
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=64,
        validation_data=(X_val, y_val)
    )

    test_loss, test_acc = model.evaluate(X_test, y_test)
    print("Test Accuracy:", test_acc)

Epoch 1/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.4706 - loss: 0.9991 - val_accuracy: 0.6395 - val_loss: 0.7963
Epoch 2/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.6729 - loss: 0.7379 - val_accuracy: 0.6675 - val_loss: 0.7696
Epoch 3/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.7514 - loss: 0.6002 - val_accuracy: 0.6350 - val_loss: 0.8266
Epoch 4/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.8211 - loss: 0.4671 - val_accuracy: 0.6200 - val_loss: 0.9127
Epoch 5/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8830 - loss: 0.3289 - val_accuracy: 0.6055 - val_loss: 1.0611
384/384 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5525 - loss: 1.2020
Test Accuracy: 0.5485997796058655


In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 50, 32)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 973,067 (3.71 MB)

 Trainable params: 324,355 (1.24 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 648,712 (2.47 MB)

## 3.2 GRU & Bidirectional GRU

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

# GRU Classifier

with tf.device('/GPU:0'):
    model = Sequential([
        Embedding(input_dim=num_words, output_dim=32, input_length=max_len),
        GRU(32),
        Dropout(0.5), # Dropout for regularization
        Dense(3, activation='softmax') # 3 classes, softmax output
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=64,
        validation_data=(X_val, y_val)
    )

    test_loss, test_acc = model.evaluate(X_test, y_test)
    print("Test Accuracy:", test_acc)

Epoch 1/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.4437 - loss: 1.0276 - val_accuracy: 0.4345 - val_loss: 1.0182
Epoch 2/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.4477 - loss: 1.0182 - val_accuracy: 0.4345 - val_loss: 1.0185
Epoch 3/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4518 - loss: 1.0133 - val_accuracy: 0.4345 - val_loss: 1.0181
Epoch 4/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.4537 - loss: 1.0172 - val_accuracy: 0.4345 - val_loss: 1.0195
Epoch 5/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.4528 - loss: 1.0161 - val_accuracy: 0.4345 - val_loss: 1.0182
384/384 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4899 - loss: 1.1572
Test Accuracy: 0.48331162333488464


In [ ]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 50, 32)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 32)             │         6,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 979,307 (3.74 MB)

 Trainable params: 326,435 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 652,872 (2.49 MB)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Bidirectional

# Bidirectional-GRU Classifier

with tf.device('/GPU:0'):
    model = Sequential([
        Embedding(input_dim=num_words, output_dim=32, input_length=max_len),
        Bidirectional(GRU(32)),
        Dropout(0.5), # Dropout for regularization
        Dense(3, activation='softmax') # 3 classes, softmax output
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=64,
        validation_data=(X_val, y_val)
    )

    test_loss, test_acc = model.evaluate(X_test, y_test)
    print("Test Accuracy:", test_acc)

Epoch 1/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.5117 - loss: 0.9706 - val_accuracy: 0.6155 - val_loss: 0.8227
Epoch 2/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.6696 - loss: 0.7468 - val_accuracy: 0.6650 - val_loss: 0.7401
Epoch 3/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.7286 - loss: 0.6348 - val_accuracy: 0.6790 - val_loss: 0.7533
Epoch 4/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7585 - loss: 0.5734 - val_accuracy: 0.6710 - val_loss: 0.7565
Epoch 5/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.7861 - loss: 0.5247 - val_accuracy: 0.6725 - val_loss: 0.7837
384/384 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5903 - loss: 0.9540
Test Accuracy: 0.5868609547615051


In [ ]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 50, 32)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        12,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 998,603 (3.81 MB)

 Trainable params: 332,867 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 665,736 (2.54 MB)

## 3.3 LSTM & Bidirectional LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# LSTM Classifier

with tf.device('/GPU:0'):
    model = Sequential([
        Embedding(input_dim=num_words, output_dim=32, input_length=max_len),
        LSTM(32),
        Dropout(0.5), # Dropout for regularization
        Dense(3, activation='softmax') # 3 classes, softmax output
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=64,
        validation_data=(X_val, y_val)
    )

    test_loss, test_acc = model.evaluate(X_test, y_test)
    print("Test Accuracy:", test_acc)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


713/713 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.4396 - loss: 1.0263 - val_accuracy: 0.4230 - val_loss: 0.9822
Epoch 2/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.4798 - loss: 0.9262 - val_accuracy: 0.6365 - val_loss: 0.8017
Epoch 3/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.6821 - loss: 0.7252 - val_accuracy: 0.6720 - val_loss: 0.7540
Epoch 4/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7324 - loss: 0.6409 - val_accuracy: 0.6585 - val_loss: 0.7839
Epoch 5/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7548 - loss: 0.5895 - val_accuracy: 0.6680 - val_loss: 0.7722
384/384 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5795 - loss: 0.9297
Test Accuracy: 0.5771735310554504


In [ ]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 50, 32)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 985,259 (3.76 MB)

 Trainable params: 328,419 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 656,840 (2.51 MB)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

# LSTM Classifier

with tf.device('/GPU:0'):
    model = Sequential([
        Embedding(input_dim=num_words, output_dim=32, input_length=max_len),
        Bidirectional(LSTM(32)),
        Dropout(0.5), # Dropout for regularization
        Dense(3, activation='softmax') # 3 classes, softmax output
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=64,
        validation_data=(X_val, y_val)
    )

    test_loss, test_acc = model.evaluate(X_test, y_test)
    print("Test Accuracy:", test_acc)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


713/713 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.5188 - loss: 0.9496 - val_accuracy: 0.6575 - val_loss: 0.7597
Epoch 2/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.7003 - loss: 0.6864 - val_accuracy: 0.6705 - val_loss: 0.7426
Epoch 3/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.7441 - loss: 0.6034 - val_accuracy: 0.6825 - val_loss: 0.7549
Epoch 4/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.7707 - loss: 0.5467 - val_accuracy: 0.6800 - val_loss: 0.7639
Epoch 5/5
713/713 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7962 - loss: 0.4949 - val_accuracy: 0.6620 - val_loss: 0.8411
384/384 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5803 - loss: 1.0048
Test Accuracy: 0.5797785520553589


In [ ]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 50, 32)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,010,507 (3.85 MB)

 Trainable params: 336,835 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 673,672 (2.57 MB)